This notebook builds stacked histograms using VAMP-seq and SGE data seen in figure 2.

In [ ]:
import pandas as pd
import altair as alt
import numpy as np

In [ ]:
#Paths to each of the SGE and VAMP-seq subsets from the main dataframe
sge_ppj_subset = pd.read_excel('../Data/filtered_ppj_data/20251202_SGEsubset.xlsx')
vampseq_ppj_subset = pd.read_excel('../Data/filtered_ppj_data/20251204_VAMPseqsubset_wDups.xlsx')

alt.data_transformers.disable_max_rows() #gets rid of max data length problem

In [ ]:
def get_gmm_threshold(df): #estimates thresholds by finding highest scoring/lowest scoring functionally abnormal/functionally normal variants respectively
    
    ab_df = df.loc[df['functional_consequence'].isin(['functionally_abnormal'])]
    norm_df = df.loc[df['functional_consequence'].isin(['functionally_normal'])]

    # now we get the scores that are the closest to the (n)ormal and (a)bnormal thresholds
    upprthresh = norm_df['score'].min()
    lwrthresh = ab_df['score'].max()


    thresholds = [lwrthresh, upprthresh]

    return thresholds

In [ ]:
def read_data(sge, vampseq): #Reads all data and changes variant consequence annotations to be more human readable
    
    sge_genes = ['BARD1', 'PALB2', 'BRCA2', 'RAD51D', 'XRCC2', 'CTCF', 'SFPQ'] #SGE genes

    vampseq_genes = ['G6PD', 'TSC2', 'F9'] #VAMP-seq genes

    sge_dict = {} #Empty dictionary to store SGE data in form or {gene_name: gene_data}

    for gene in sge_genes:
        print(f' Reading {gene}...')
        df = sge.loc[sge['Gene'] == gene]

        df = df.dropna(subset = ['simplified_consequence'])
        df = df.rename(columns = {'simplified_consequence': 'Consequence',
                                 'auth_reported_score': 'score',
                                  'auth_reported_func_class': 'functional_consequence',
                                 'ref_allele': 'ref',
                                 'clinvar_sig': 'Germline classification'})
        df.loc[df['Consequence'].str.contains('missense'), 'Consequence'] = 'Missense'
        df.loc[df['Consequence'] == 'synonymous_variant', 'Consequence'] = 'Synonymous'
        df.loc[df['Consequence'] == 'intron_variant', 'Consequence'] = 'Intron'
        df.loc[df['Consequence'] == 'stop_gained', 'Consequence'] = 'Stop Gained'
        df.loc[df['Consequence'] == 'stop_lost', 'Consequence'] = 'Stop Lost'
        df.loc[df['Consequence'].str.contains('site'), 'Consequence'] = 'Canonical Splice'
        df.loc[df['Consequence'].str.contains('ing_var'), 'Consequence'] = 'Splice Region'
        df.loc[df['Consequence'].str.contains('UTR'), 'Consequence'] = 'UTR Variant'
        df.loc[df['Consequence'] == 'start_lost', 'Consequence'] = 'Start Lost'
        df.loc[df['ref'].str.len() > 1, 'Consequence'] = '3bp Deletion'

        sge_dict[gene] = df


    #Analagous code for reading VAMP-seq data
    vamp_dict = {}
    for gene in vampseq_genes:
        #Merges TSC2 datasets into single dataframe
        if gene == 'TSC2': 
            df = vampseq.loc[vampseq['Gene'] == gene]

            rapgap_df = df.loc[df['Dataset'].str.contains('rapgap')].copy()
            tuberin_df = df.loc[df['Dataset'].str.contains('tuberin')].copy()
            tsc2_sets = [('TSC2 RapGAP', rapgap_df), ('TSC2 Tuberin', tuberin_df)]

            for set in tsc2_sets:
                name, df = set
                df['simplified_consequence'] = df['simplified_consequence'].fillna('filter')
                df = df.rename(columns = {'simplified_consequence': 'Consequence',
                                         'auth_reported_score': 'score',
                                          'auth_reported_func_class': 'functional_consequence',
                                         'ref_allele': 'ref',
                                         'clinvar_sig': 'Germline classification'})
                df.loc[df['Consequence'].str.contains('missense'), 'Consequence'] = 'Missense'
                df.loc[df['Consequence'] == 'synonymous_variant', 'Consequence'] = 'Synonymous'
                df.loc[df['Consequence'] == 'intron_variant', 'Consequence'] = 'Intron'
                df.loc[df['Consequence'] == 'stop_gained', 'Consequence'] = 'Stop Gained'
                df.loc[df['Consequence'] == 'stop_lost', 'Consequence'] = 'Stop Lost'
                df.loc[df['Consequence'].str.contains('site'), 'Consequence'] = 'Canonical Splice'
                df.loc[df['Consequence'].str.contains('ing_var'), 'Consequence'] = 'Splice Region'
                df.loc[df['Consequence'].str.contains('UTR'), 'Consequence'] = 'UTR Variant'
                df.loc[df['Consequence'] == 'start_lost', 'Consequence'] = 'Start Lost'

                vamp_dict[name] = df
        else:
            df = vampseq.loc[vampseq['Gene'] == gene]
            df['simplified_consequence'] = df['simplified_consequence'].fillna('filter')
            df = df.rename(columns = {'simplified_consequence': 'Consequence',
                                     'auth_reported_score': 'score',
                                      'auth_reported_func_class': 'functional_consequence',
                                     'ref_allele': 'ref',
                                     'clinvar_sig': 'Germline classification'})
            df.loc[df['Consequence'].str.contains('missense'), 'Consequence'] = 'Missense'
            df.loc[df['Consequence'] == 'synonymous_variant', 'Consequence'] = 'Synonymous'
            df.loc[df['Consequence'] == 'intron_variant', 'Consequence'] = 'Intron'
            df.loc[df['Consequence'] == 'stop_gained', 'Consequence'] = 'Stop Gained'
            df.loc[df['Consequence'] == 'stop_lost', 'Consequence'] = 'Stop Lost'
            df.loc[df['Consequence'].str.contains('site'), 'Consequence'] = 'Canonical Splice'
            df.loc[df['Consequence'].str.contains('ing_var'), 'Consequence'] = 'Splice Region'
            df.loc[df['Consequence'].str.contains('UTR'), 'Consequence'] = 'UTR Variant'
            df.loc[df['Consequence'] == 'start_lost', 'Consequence'] = 'Start Lost'
        
        vamp_dict[gene] = df

    return sge_dict, vamp_dict

In [ ]:
def clinvar_density_plot(df,plot_domain): #Custom function to build ClinVar density plot overlay onto histograms
    
    w_clinvar_df = (df.dropna(subset = ['Germline classification'])).copy() #Gets ClinVar variants
    
    w_clinvar_df = w_clinvar_df.loc[~w_clinvar_df['Germline classification'].isin(['Uncertain significance', 'Conflicting classifications of pathogenicity'])] #Drops VUS

    #Renames BLB and PLP variant types. All BLB and PLPs are grouped into benign and pathogenic groups respectively
    w_clinvar_df.loc[(w_clinvar_df['Germline classification'] == 'Benign') | (w_clinvar_df['Germline classification'] == 'Likely benign') | (w_clinvar_df['Germline classification'] == 'Benign/Likely benign'), 'simple_classification'] = 'Benign/Likely Benign'
    w_clinvar_df.loc[(w_clinvar_df['Germline classification'] == 'Pathogenic') | (w_clinvar_df['Germline classification'] == 'Likely pathogenic') | (w_clinvar_df['Germline classification'] == 'Pathogenic/Likely pathogenic'), 'simple_classification'] = 'Pathogenic/Likely Pathogenic'
    
    w_clinvar_df = w_clinvar_df.loc[w_clinvar_df['simple_classification'].isin(['Benign/Likely Benign',  'Pathogenic/Likely Pathogenic'])]

    #Builds density plot
    print(f' This is how many ClinVar variants there are: {len(w_clinvar_df)}')
    density_plot = alt.Chart(w_clinvar_df).transform_density(
        'score',
        as_=['score', 'density'],
        groupby=['simple_classification'],
        extent=plot_domain
    ).mark_line(strokeWidth=2).encode(
        x=alt.X('score:Q', scale=alt.Scale(domain=plot_domain)),
        y=alt.Y('density:Q', axis = None),
        color=alt.Color('simple_classification:N',
                       scale = alt.Scale(
                           domain = ['Pathogenic/Likely Pathogenic', 'Benign/Likely Benign'],
                           range = ['#CA7682','#1D7AAB']
                       ),
                      legend = None
                       )
    )

    #Extra plot for legend 
    w_legend_density_plot = alt.Chart(w_clinvar_df).transform_density(
        'score',
        as_=['score', 'density'],
        groupby=['simple_classification'],
        extent=plot_domain
    ).mark_line(strokeWidth=2).encode(
        x=alt.X('score:Q', scale=alt.Scale(domain=plot_domain)),
        y=alt.Y('density:Q', axis = None),
        color=alt.Color('simple_classification:N',
                       scale = alt.Scale(
                           domain = ['Pathogenic/Likely Pathogenic', 'Benign/Likely Benign'],
                           range = ['#CA7682','#1D7AAB']
                       ),
                        legend = alt.Legend(
                            title = 'ClinVar Classification',
                            titleFontSize = 14,
                            labelFontSize = 12
                        )
        )
    )

    #w_legend_density_plot.display()
    return density_plot

In [ ]:
def sge_stacked_histos(sge_genes, show_inset=False):
    """
    Builds stacked histograms for SGE genes in two columns (4 left, 3 right).
    
    Args:
        sge_genes: dict of gene dataframes
        show_inset: if True, creates inset plots for functionally abnormal variants
    
    Returns:
        final_plot if show_inset=False
        (final_plot, inset_dict) if show_inset=True
    """

    genes = list(sge_genes.keys())
    
    # Split into two columns
    left_genes = genes[:4]
    right_genes = genes[4:]

    # Dimensions of each individual plot
    std_height = 150
    std_width = 500
    std_dy = 35
    
    palette = [
        '#006616',  # dark green
        '#81B4C7',  # dusty blue
        '#ffcd3a',  # yellow
        '#6AA84F',  # med green
        '#93C47D',  # light green
        '#888888',  # med gray
        '#000000',  # black
        '#1170AA',  # darker blue
        '#CFCFCF'   # light gray  
    ]
    
    variant_types = [
        'Synonymous',
        'Missense',  
        'Stop Gained',
        'Intron', 
        'UTR Variant',
        'Stop Lost',
        'Start Lost',
        'Canonical Splice', 
        'Splice Region'
    ]

    plot_domain = [-0.45, 0.1]
    tick_values = [-0.4, -0.3, -0.2, -0.1, 0, 0.1]
    title_fontsize = 26
    
    # Store inset plots if needed
    inset_plots = {}
    
    def build_inset(gene, df, thresholds):
        """Build a small inset histogram for functionally abnormal variants."""
        
        # For genes without functional_consequence (like BRCA2), use score < -0.1
        if thresholds is None:
            df_abnormal = df[df['score'] < -0.1].copy()
        else:
            df_abnormal = df[df['functional_consequence'] == 'functionally_abnormal'].copy()
        
        if len(df_abnormal) == 0:
            return None
        
        inset = alt.Chart(df_abnormal).mark_bar().encode(
            alt.X('score:Q',
                  bin=True,
                  axis=alt.Axis(title='', labelFontSize=9, tickSize=2, tickCount=3),
                  scale=alt.Scale(nice=False)),
            alt.Y('count():Q',
                  axis=alt.Axis(title='', labelFontSize=9, tickSize=2, tickCount=3),
                  scale=alt.Scale(zero=True)),
            color=alt.Color('Consequence:N',
                            scale=alt.Scale(range=palette, domain=variant_types),
                            legend=None)
        ).properties(
            width=120,
            height=36,
            title=alt.TitleParams(
                text=f'Abnormal (n={len(df_abnormal)})',
                fontSize=10,
                anchor='start'
            )
        ).configure_axis(
            grid=False
        ).configure_view(
            stroke=None
        )

        inset.display()
        
        return inset
        
    def build_histogram(gene, is_first, is_bottom, is_left_column):
        """Build a single histogram with appropriate axis/legend settings."""
        df = sge_genes[gene]
        length = len(df)

        # Filter to plot domain
        df = df.loc[(df['score'] >= plot_domain[0]) & (df['score'] <= plot_domain[1])].copy()
        
        # Determine thresholds
        if gene != 'BRCA2':
            thresholds = get_gmm_threshold(df)
        else:
            thresholds = None
        
        # --- Build axis configurations based on position ---
        
        # X-axis: only show labels on bottom charts of each column
        if is_bottom:
            x_axis = alt.Axis(title='Fitness Score', titleFontSize=24, labelFontSize=22, 
                              tickSize=3, tickWidth=3, values=tick_values)
        else:
            x_axis = alt.Axis(title='', labels=False, tickSize=3, tickWidth=3, values=tick_values)
        
        # Y-axis: only show title on left column
        if is_left_column:
            y_axis = alt.Axis(title='# of Vars.' if is_first else '', 
                              labelFontSize=22, tickSize=3, tickWidth=3, titleFontSize=24)
        else:
            y_axis = alt.Axis(title='', labelFontSize=22, tickSize=3, tickWidth=3, titleFontSize=24)
        
        # Legend
        show_legend = None
        
        # --- Build histogram ---
        
        chart = alt.Chart(df).mark_bar().encode(
            alt.X('score:Q', axis=x_axis, scale=alt.Scale(domain=plot_domain), bin=alt.Bin(maxbins=50)),
            alt.Y('count():Q', axis=y_axis, scale=alt.Scale(zero=False)),
            color=alt.Color('Consequence:N', scale=alt.Scale(range=palette, domain=variant_types), 
                            legend=show_legend)
        ).properties(
            height=std_height, width=std_width,
            title=alt.TitleParams(text=f' {gene} (n = {length:,d})', fontSize=title_fontsize, dy=std_dy)
        )
        
        # --- Add threshold lines and density plots ---
        
        layers = [chart]
        
        if thresholds is not None:
            nf_line = alt.Chart(pd.DataFrame({'Functional Score': [thresholds[0]]})).mark_rule(
                color='black', strokeDash=[8, 8], strokeWidth=2
            ).encode(x='Functional Score:Q')
            
            func_line = alt.Chart(pd.DataFrame({'Functional Score': [thresholds[1]]})).mark_rule(
                color='#888888', strokeDash=[8, 8], strokeWidth=2
            ).encode(x='Functional Score:Q')
            
            layers.extend([nf_line, func_line])
        
        # Add ClinVar density plot (except for SFPQ)
        if gene != 'SFPQ':
            clinvar_density = clinvar_density_plot(df, plot_domain)
            layers.append(clinvar_density)
            
            chart = alt.layer(*layers).resolve_scale(y='independent', color='independent')
        else:
            chart = alt.layer(*layers)
        
        # Build inset if requested
        if show_inset:
            inset = build_inset(gene, df, thresholds)
            if inset is not None:
                inset_plots[gene] = inset
        
        return chart
    
    # --- Build left column (4 genes) ---
    left_histograms = []
    for i, gene in enumerate(left_genes):
        is_first = (i == 0)
        is_bottom = (i == len(left_genes) - 1)
        chart = build_histogram(gene, is_first, is_bottom, is_left_column=True)
        left_histograms.append(chart)
    
    left_column = left_histograms[0]
    for chart in left_histograms[1:]:
        left_column = alt.vconcat(left_column, chart, spacing=-5)
    
    # --- Build right column (3 genes) ---
    right_histograms = []
    for i, gene in enumerate(right_genes):
        is_first = False
        is_bottom = (i == len(right_genes) - 1)
        chart = build_histogram(gene, is_first, is_bottom, is_left_column=False)
        right_histograms.append(chart)
    
    right_column = right_histograms[0]
    for chart in right_histograms[1:]:
        right_column = alt.vconcat(right_column, chart, spacing=-5)
    
    # --- Combine columns horizontally ---
    final_plot = alt.hconcat(left_column, right_column, spacing=20)
    
    final_plot = final_plot.configure_axis(
        grid=False
    ).configure_view(
        stroke=None
    ).resolve_scale(
        x='shared',
        y='independent'
    )
    
    # --- Handle return ---
    if show_inset:
        return final_plot, inset_plots
    
    return final_plot

In [ ]:
def vampseq_stacked_histos(vampseq_data, thresholding=False):
    """Stacked histograms for VAMP-seq genes in 2x2 grid"""
    
    genes = list(vampseq_data.keys())
    print(genes)
    std_height = 262.5
    std_width = 500
    title_fontsize = 26
    
    palette = [
        '#006616',  # dark green
        '#81B4C7',  # dusty blue
        '#ffcd3a'   # yellow  
    ]
    
    variant_types = [
        'Synonymous',
        'Missense',  
        'Stop Gained'
    ]
    
    plot_domain = [-0.25, 1.5]
    tick_values = [-0.25, 0, 0.25, 0.50, 0.75, 1, 1.25, 1.5]
    
    # Genes to skip
    skip_genes = ['F9 Ab001', 'F9 Ab124', 'F9 Ab3570', 'F9 Strep', 'TSC2']
    
    def build_histogram(gene, df_dup, is_first, is_bottom, is_left_column):
        """Build a single histogram with appropriate axis/legend settings."""
        
        df = df_dup.drop_duplicates(subset=['ID'])
        length = len(df)
        
        df = df.dropna(subset=['Consequence'])
        df = df.loc[(df['score'] >= plot_domain[0]) & (df['score'] <= plot_domain[1])]
        thresholds = get_gmm_threshold(df)
        
        # Determine axis title based on position
        x_axis_title = 'Abundance Score' if is_bottom else ''
        score_label = 'Abundance Score' if is_bottom else 'Functional Score'
        
        # X-axis: labels only on bottom charts
        if is_bottom:
            x_axis = alt.Axis(title=x_axis_title, titleFontSize=24, labelFontSize=22,
                              tickSize=3, tickWidth=3, values=tick_values, format = '.2f')
        else:
            x_axis = alt.Axis(title='', labels=False, tickSize=3, tickWidth=3, values=tick_values)
        
        # Y-axis: title only on top-left
        if is_left_column and is_first:
            y_axis = alt.Axis(title='# of Vars.', labelFontSize=22, titleFontSize=24)
        else:
            y_axis = alt.Axis(title='', tickCount=3, tickMinStep=1, labelFontSize=22, titleFontSize=24)
        
        # Legend: only on first chart (top-left)
        show_legend = alt.Legend(titleFontSize=18, labelFontSize=16, orient='top-left',
                                  columns=1, offset=10) if is_first else None
        
        # Custom title for F9
        if gene == 'F9':
            title_text = f' F9 (Heavy-Chain Ab) (n = {length:,d})'
        else:
            title_text = f' {gene} (n = {length:,d})'
        
        # Build histogram
        chart = alt.Chart(df).mark_bar().encode(
            alt.X('score', axis=x_axis, scale=alt.Scale(domain=plot_domain), bin=alt.Bin(maxbins=50)),
            alt.Y('count()', axis=y_axis, scale=alt.Scale(nice=False, zero=False)),
            color=alt.Color('Consequence:N', scale=alt.Scale(range=palette, domain=variant_types),
                            legend=show_legend)
        ).properties(
            height=std_height,
            width=std_width,
            title=alt.TitleParams(text=title_text, fontSize=title_fontsize, dy=15)
        )
        
        # Add ClinVar density
        clinvar_density = clinvar_density_plot(df, plot_domain)
        chart = alt.layer(chart, clinvar_density).resolve_scale(y='independent', color='independent')
        
        # Add threshold lines if enabled
        if thresholding:
            nf_line = alt.Chart(pd.DataFrame({score_label: [thresholds[0]]})).mark_rule(
                color='black', strokeDash=[8, 8], strokeWidth=2
            ).encode(x=f'{score_label}:Q')
            
            func_line = alt.Chart(pd.DataFrame({score_label: [thresholds[1]]})).mark_rule(
                color='#888888', strokeDash=[8, 8], strokeWidth=2
            ).encode(x=f'{score_label}:Q')
            
            chart = alt.layer(chart, nf_line, func_line)
        
        return chart
    
    # Collect histograms (skipping specified genes)
    histograms = []
    for gene in genes:
        if gene in skip_genes:
            continue
        histograms.append((gene, vampseq_data[gene]))
    
    # Split into 2x2 grid (left column: first 2, right column: last 2)
    left_items = histograms[:2]
    right_items = histograms[2:]
    
    # Build left column
    left_histograms = []
    for i, (gene, df_dup) in enumerate(left_items):
        is_first = (i == 0)
        is_bottom = (i == len(left_items) - 1)
        chart = build_histogram(gene, df_dup, is_first, is_bottom, is_left_column=True)
        left_histograms.append(chart)
    
    left_column = left_histograms[0]
    for chart in left_histograms[1:]:
        left_column = alt.vconcat(left_column, chart, spacing=-5)
    
    # Build right column
    right_histograms = []
    for i, (gene, df_dup) in enumerate(right_items):
        is_first = False  # Legend only on top-left
        is_bottom = (i == len(right_items) - 1)
        chart = build_histogram(gene, df_dup, is_first, is_bottom, is_left_column=False)
        right_histograms.append(chart)
    
    right_column = right_histograms[0]
    for chart in right_histograms[1:]:
        right_column = alt.vconcat(right_column, chart, spacing=-5)
    
    # Combine columns horizontally
    final_plot = alt.hconcat(left_column, right_column, spacing=20)
    
    final_plot = final_plot.configure_axis(
        grid=False
    ).configure_view(
        stroke=None
    ).resolve_scale(
        x='shared',
        y='independent'
    )

    final_plot.display()
    return final_plot

In [ ]:
def main():
    sge_data, vampseq_data = read_data(sge_ppj_subset, vampseq_ppj_subset)

    #Filters out 3bp-deletions in the XRCC2 dataset
    new_xrcc2 = sge_data['XRCC2']
    new_xrcc2 = new_xrcc2.loc[new_xrcc2['ref'].str.len() == 1]
    sge_data['XRCC2'] = new_xrcc2

    stacked_histos, insets_dict = sge_stacked_histos(sge_data, show_inset = True)
    stacked_histos.display()
    
    vampseq_histos = vampseq_stacked_histos(vampseq_data, thresholding = True)
    
    save = True
    if save:
        stacked_histos.save('/Users/ivan/Desktop/pillar_project_figs/Histogram_wStripplot/20260120all_sge_histograms.svg')
        vampseq_histos.save('/Users/ivan/Desktop/pillar_project_figs/Histogram_wStripplot/20260120all_vampseq_histograms_bigger.svg')

        # Save individual insets
        for gene, inset in insets_dict.items():
            inset.save(f'/Users/ivan/Desktop/pillar_project_figs/20260120_sge_histogram_inset_{gene}.svg')

In [ ]:
main()